# SUSA-1B CodeMind — Google Colab Training Notebook

Welcome to the official training runner for **SUSA-1B CodeMind** under the SUSA technology ecosystem.

### Pipeline Overview
1. **Environment Setup & Verification:** Connects to runtime, inspects GPU/VRAM, clones the repository.
2. **Safety Check:** Runs analytical resource estimation against allocated VRAM.
3. **Drive Mount:** Optional Google Drive synchronization for persistent checkpoints.
4. **Benchmark & Smoke Test:** Validates forward/backward pass before training starts.
5. **Execution:** Runs pretraining with checkpointing and resume support.

### 1. Repository Setup & Clone

In [ ]:
# Clone or pull the approved repository
import os

REPO_URL = input("Enter your GitHub repository URL (e.g. https://github.com/your-username/susa-1b-codemind.git): ").strip()
REPO_NAME = "susa-1b-codemind"

if os.path.exists(REPO_NAME):
    print(f"Pulling latest changes in {REPO_NAME}...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_NAME}

%cd {REPO_NAME}
!pwd

### 2. Install Documented Dependencies

In [ ]:
# Install project in editable mode with colab dependencies
!pip install --upgrade pip
!pip install -e ".[colab,dev]"
!pip install psutil pyyaml

### 3. GPU Hardware Diagnostics & Verification

In [ ]:
import torch
from susacodemind.utils.system_checker import SystemInfo

info = SystemInfo.inspect()
print(info.summary())

if not info.cuda_available:
    print("\n[WARNING] No CUDA GPU detected! Free Colab session may need runtime type changed to GPU:")
    print("Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")
    print("Stopping before expensive training.")

### 4. Select Profile & Verify Resource Feasibility

In [ ]:
# Choose profile: smoke, small, medium, or target_1b
PROFILE = "small" #@param ["smoke", "small", "medium", "target_1b"]

from susacodemind.model.config import ModelConfig
from susacodemind.utils.resource_estimator import estimate_resources

config_path = f"configs/{PROFILE}.yaml"
cfg = ModelConfig.from_yaml(config_path)
print(cfg.print_parameter_summary())

est = estimate_resources(cfg, batch_size=2, seq_len=cfg.max_seq_len)
print(est.summary())

### 5. Optional Google Drive Mount (For Persistent Checkpoints)

In [ ]:
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/SUSA_CodeMind_Checkpoints"

if USE_DRIVE:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print(f"Google Drive checkpoint directory ready at: {DRIVE_CHECKPOINT_DIR}")
else:
    print("Using local runtime storage only.")

### 6. Run Smoke Tests & Verification Suite

In [ ]:
!pytest tests/ -v